# Shadow Prices in passagemath

**What this notebook teaches:** the workflow for reading dual values (shadow prices) from a GLPK LP solve, and the one trap that gives wrong answers silently.

**Why it matters:** dual values aren't documented in `linear_programming.rst`. After this notebook you'll understand the workflow well enough to write that section.

**Structure:** build → find → understand → trap → exercise. Run cells in order.

In [ ]:
# Setup — run this first.
from sage.numerical.mip import MixedIntegerLinearProgram
print("OK")

---
## Section 1: Build and solve the LP

Three constraints, two variables:

- maximize `3*x[0] + 2*x[1]`
- `x[0] + x[1] <= 4`  (shared resource)
- `x[0] <= 2`
- `x[1] <= 3`

The `simplex_or_intopt` parameter matters — see Section 4.

In [ ]:
p = MixedIntegerLinearProgram(maximization=True, solver='GLPK')
x = p.new_variable(nonnegative=True)

p.set_objective(3*x[0] + 2*x[1])
p.add_constraint(x[0] + x[1] <= 4)  # row 0
p.add_constraint(x[0] <= 2)          # row 1
p.add_constraint(x[1] <= 3)          # row 2

p.solver_parameter('simplex_or_intopt', 'simplex_only')
opt = p.solve()
vals = p.get_values(x)

print(f"objective: {opt}")
print(f"x[0] = {vals[0]},  x[1] = {vals[1]}")

In [ ]:
# Which constraints are binding?
activities = [
    vals[0] + vals[1],  # row 0: x[0] + x[1]
    vals[0],            # row 1: x[0]
    vals[1],            # row 2: x[1]
]
rhs = [4, 2, 3]
for i in range(3):
    if activities[i] == rhs[i]:
        status = 'binding'
    else:
        status = f"slack  (used {activities[i]:.0f} of {rhs[i]})"
    print(f"row {i}: {status}")

Rows 0 and 1 are binding. Row 2 is slack — `x[1] = 2` is below its cap of 3.

Relaxing row 2 changes nothing. The leverage is on rows 0 and 1.

Shadow prices (dual values) quantify exactly how much: one number per constraint, telling you how much the objective improves if you relax that constraint by one unit.

---
## Section 2: Find the dual values

In [ ]:
# There's no high-level method on the frontend:
try:
    p.get_dual_values()
except AttributeError as e:
    print(f"AttributeError: {e}")

In [ ]:
# They live on the backend.
b = p.get_backend()
print("backend:", type(b).__name__)
print()
for i in range(3):
    print(f"row {i} dual: {b.get_row_dual(i)}")

Row 2 is 0 — consistent with it being slack. Rows 0 and 1 are non-zero.

But what does 2.0 mean? And 1.0?

---
## Section 3: What the numbers mean

Claim: `get_row_dual(0) = 2.0` means relaxing the first constraint by one unit improves the objective by 2.

Test it:

In [ ]:
# Increase row 0's RHS from 4 to 5 — self-contained rebuild
_p1 = MixedIntegerLinearProgram(maximization=True, solver='GLPK')
_x1 = _p1.new_variable(nonnegative=True)
_p1.set_objective(3*_x1[0] + 2*_x1[1])
_p1.add_constraint(_x1[0] + _x1[1] <= 5)  # was 4
_p1.add_constraint(_x1[0] <= 2)
_p1.add_constraint(_x1[1] <= 3)
_p1.solver_parameter('simplex_or_intopt', 'simplex_only')
_opt1 = _p1.solve()
print(f"original: {opt},  relaxed row 0 by 1: {_opt1},  difference: {_opt1 - opt}")
print(f"get_row_dual(0) = {b.get_row_dual(0)}")
print(f"match: {_opt1 - opt == b.get_row_dual(0)}")

In [ ]:
# Check all three rows the same way
_base_rhs = [4, 2, 3]
for i in range(3):
    _p2 = MixedIntegerLinearProgram(maximization=True, solver='GLPK')
    _x2 = _p2.new_variable(nonnegative=True)
    _p2.set_objective(3*_x2[0] + 2*_x2[1])
    _r = _base_rhs[:]
    _r[i] += 1
    _p2.add_constraint(_x2[0] + _x2[1] <= _r[0])
    _p2.add_constraint(_x2[0] <= _r[1])
    _p2.add_constraint(_x2[1] <= _r[2])
    _p2.solver_parameter('simplex_or_intopt', 'simplex_only')
    _opt2 = _p2.solve()
    change = _opt2 - opt
    dual = b.get_row_dual(i)
    print(f"row {i}: Δobjective = {change:.1f},  dual = {dual:.1f},  match = {change == dual}")

`get_row_dual(i)` = how much the objective improves per unit increase in constraint `i`'s RHS.

Zero for row 2 because it's already slack — relaxing it does nothing. Rows 0 and 1 are genuinely scarce resources.

Also confirmed: row index = order of `add_constraint` calls, 0-based.

---
## Section 4: Plot twist

The default GLPK mode is `intopt`. It gets the correct objective value and variable values. But it does not compute dual variables.

If you call `get_row_dual` after an `intopt` solve, you get zeros. No exception. Just silently wrong.

In [ ]:
# Same LP — no simplex_only
_pt = MixedIntegerLinearProgram(maximization=True, solver='GLPK')
_xt = _pt.new_variable(nonnegative=True)
_pt.set_objective(3*_xt[0] + 2*_xt[1])
_pt.add_constraint(_xt[0] + _xt[1] <= 4)
_pt.add_constraint(_xt[0] <= 2)
_pt.add_constraint(_xt[1] <= 3)
# no solver_parameter call — uses intopt

_opt_t = _pt.solve()
_bt = _pt.get_backend()
_vals_t = _pt.get_values(_xt)

print(f"objective: {_opt_t}  (correct)")
print(f"x[0] = {_vals_t[0]}  (correct)")
print()
for i in range(3):
    print(f"row {i} dual: {_bt.get_row_dual(i)}")  # silently wrong

Objective is 10.0, `x[0]` is 2.0 — both correct. The duals are all zero — all wrong.

This is the most practical thing to document: dual values require `simplex_only` mode. Omitting it gives wrong answers with no error.

---
## Section 5: The complete workflow

Copy-paste ready:

In [ ]:
# Shadow prices with GLPK — complete pattern
_pw = MixedIntegerLinearProgram(maximization=True, solver='GLPK')
_xw = _pw.new_variable(nonnegative=True)

_pw.set_objective(3*_xw[0] + 2*_xw[1])
_pw.add_constraint(_xw[0] + _xw[1] <= 4)  # row 0
_pw.add_constraint(_xw[0] <= 2)             # row 1
_pw.add_constraint(_xw[1] <= 3)             # row 2

_pw.solver_parameter('simplex_or_intopt', 'simplex_only')  # required for duals
_pw.solve()

_bw = _pw.get_backend()  # duals live on the backend, not the frontend
for i in range(3):
    print(f"row {i} shadow price: {_bw.get_row_dual(i)}")

Three things to get right:

1. Use `solver='GLPK'` (or another backend that implements `get_row_dual`)
2. Set `simplex_or_intopt` to `'simplex_only'` **before** solving
3. Row indices are 0-based, in the order constraints were added

---
## Section 6: Exercise

Different LP, same question.

A shop sells two items with revenues `2*B + 3*C`, subject to:

- `B + C <= 4`  (total stock limit)
- `B <= 3`       (supply cap on B)
- `C <= 3`       (supply cap on C)

Which constraint is most valuable to relax? Set up the full workflow: solve in simplex mode, read the duals, and verify one of them with a perturbation.

In [ ]:
# Your work here


Work it out above, then check the answer below.

---
### Answer

In [ ]:
# Answer — self-contained
_pe = MixedIntegerLinearProgram(maximization=True, solver='GLPK')
_ue = _pe.new_variable(nonnegative=True)

_pe.set_objective(2*_ue[0] + 3*_ue[1])
_pe.add_constraint(_ue[0] + _ue[1] <= 4)  # row 0: total stock
_pe.add_constraint(_ue[0] <= 3)             # row 1: cap on B
_pe.add_constraint(_ue[1] <= 3)             # row 2: cap on C

_pe.solver_parameter('simplex_or_intopt', 'simplex_only')
_opt_e = _pe.solve()
_vals_e = _pe.get_values(_ue)
_be = _pe.get_backend()

print(f"objective: {_opt_e}")
print(f"B = {_vals_e[0]},  C = {_vals_e[1]}")
print()
for i in range(3):
    print(f"row {i} dual: {_be.get_row_dual(i)}")

print()
# Verify row 0: increase total stock limit from 4 to 5
_pe2 = MixedIntegerLinearProgram(maximization=True, solver='GLPK')
_ue2 = _pe2.new_variable(nonnegative=True)
_pe2.set_objective(2*_ue2[0] + 3*_ue2[1])
_pe2.add_constraint(_ue2[0] + _ue2[1] <= 5)  # +1
_pe2.add_constraint(_ue2[0] <= 3)
_pe2.add_constraint(_ue2[1] <= 3)
_pe2.solver_parameter('simplex_or_intopt', 'simplex_only')
_opt_e2 = _pe2.solve()
print(f"Relax row 0 by 1: {_opt_e} → {_opt_e2}  (Δ = {_opt_e2 - _opt_e})")
print(f"Matches get_row_dual(0) = {_be.get_row_dual(0)}: {_opt_e2 - _opt_e == _be.get_row_dual(0)}")

---
## Section 7: Write the contribution

`linear_programming.rst` has no coverage of dual values. Verify that before using the draft below:

```bash
grep -n 'dual\|shadow' src/doc/en/thematic_tutorials/linear_programming.rst
```

(No output.)

The new section belongs at line 440 of that file, between the Flow example and the Solvers section.
Here is a draft. Every claim in it was verified experimentally in this notebook.

In [ ]:
rst_draft = r'''
Shadow Prices
~~~~~~~~~~~~~

After solving an LP with the GLPK backend in simplex mode, the *shadow
price* (dual variable) of each constraint is available via
:meth:`~sage.numerical.backends.glpk_backend.GLPKBackend.get_row_dual`.
The shadow price of constraint `i` is the amount by which the objective
would improve if that constraint's right-hand side were relaxed by one unit.

Set the solver to simplex mode **before** calling :meth:`solve`::

    sage: p = MixedIntegerLinearProgram(maximization=True, solver='GLPK')
    sage: x = p.new_variable(nonnegative=True)
    sage: p.set_objective(3*x[0] + 2*x[1])
    sage: p.add_constraint(x[0] + x[1] <= 4)  # row 0
    sage: p.add_constraint(x[0] <= 2)          # row 1
    sage: p.add_constraint(x[1] <= 3)          # row 2
    sage: p.solver_parameter('simplex_or_intopt', 'simplex_only')
    sage: p.solve()
    10.0
    sage: b = p.get_backend()
    sage: b.get_row_dual(0)
    2.0
    sage: b.get_row_dual(1)
    1.0
    sage: b.get_row_dual(2)
    0.0

Row `i` corresponds to the `i`-th constraint added (0-indexed). Row 2 has
shadow price 0 because its constraint ``x[1] <= 3`` is not binding at the
optimum --- ``x[1] = 2 < 3``.

.. WARNING::

    The default solver mode is ``simplex_then_intopt``. Using it causes
    :meth:`get_row_dual` to return ``0.0`` for all rows with no warning
    or error. Shadow prices are only valid after a simplex-mode solve.
'''
print(rst_draft)

Every claim in the draft traces back to a specific cell in this notebook:

| Claim | Cell |
|---|---|
| `get_row_dual` is on the backend, not the frontend | 6 — `AttributeError` on `p.get_dual_values()` |
| Row 0 shadow price = 2.0 | 9 — perturbation: Δobj = 2.0 |
| Row 2 shadow price = 0, constraint slack | 4 — binding check |
| Row indices follow `add_constraint` order | 9–10 — all three perturbations match |
| `intopt` mode gives 0.0 silently | 12–13 — plot twist |

If you add anything to the draft, apply the same filter: *was this verified in code?*